In [1]:
%matplotlib inline
import os
import pandas as pd
import glob
import matplotlib.pyplot as plt

# 定义基础路径
base_path = 'C:/Users/snong/Desktop/remote_data/green/'

# 自动获取目录下所有的 CSV 文件
# 假设该目录下只有一个你要处理的 csv 文件
search_pattern = os.path.join(base_path, '*.csv')
found_files = glob.glob(search_pattern)

if not found_files:
    print(f"⚠️ 警告: 在 {base_path} 目录下没有找到任何 CSV 文件！")
else:
    # 直接获取找到的第一个文件
    file_path = found_files[0]
    file_name = os.path.basename(file_path)
    print(f"正在处理唯一找到的文件: {file_name} ...")

    all_parsed_rows = []
    current_date, current_time, current_offset = "", "", ""
    
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                line = line.strip()
                
                # 忽略空行和 '}'
                if not line or line == '}':
                    continue
                
                # 提取时间戳逻辑
                if ',' in line:
                    parts = line.split(',', 3)
                    current_date = parts[0].strip() if len(parts) > 0 else ""
                    current_time = parts[1].strip() if len(parts) > 1 else ""
                    current_offset = parts[2].strip() if len(parts) > 2 else ""
                    # 增加健壮性，防止某些行只有逗号没有数据
                    data_part = parts[3].strip() if len(parts) > 3 else "" 
                else:
                    data_part = line
                    
                row_dict = {
                    'File_Source': file_name,  # 添加文件来源标识
                    'Date': current_date,
                    'Time': current_time,
                    'Duration': current_offset
                }
                
                # 提取各个传感器数据
                if data_part:
                    kv_pairs = data_part.split(';')
                    for kv in kv_pairs:
                        if ':' in kv:
                            key, value = kv.split(':', 1)
                            row_dict[key.strip()] = value.strip()
                            
                    all_parsed_rows.append(row_dict)
                    
    except Exception as e:
        print(f"读取文件时发生错误: {e}")

    # 检查是否解析到了数据
    if all_parsed_rows:
        # 转换为 DataFrame
        df = pd.DataFrame(all_parsed_rows)
        print(f"\n✅ 数据解析完成！DataFrame 总行数: {len(df)}")
        
        # 动态生成保存的文件名，例如原文件叫 "test.csv"，这里存为 "test_parsed.csv"
        output_file_name = file_name.replace('.csv', '_parsed.csv')
        output_path = os.path.join(base_path, output_file_name)
        
        df.to_csv(output_path, index=False, encoding='utf-8')
        print(f"✅ 转换后的数据已保存至: {output_path}")
    else:
        print("⚠️ 未从文件中提取到任何有效数据。")

df.drop("File_Source", inplace=True, axis=1)



正在处理唯一找到的文件: 06101131_wave_heart_SpO2_data_2026-06-10T11%3A31%.csv ...

✅ 数据解析完成！DataFrame 总行数: 581710
✅ 转换后的数据已保存至: C:/Users/snong/Desktop/remote_data/green/06101131_wave_heart_SpO2_data_2026-06-10T11%3A31%_parsed.csv


In [2]:
df.columns = ['Date', 'Time', 'Duration', 'green1', 'green2', 'ir1', 'ir2', 'accX', 'accY', 'accZ']
# 指定需要转换的列名
cols_to_convert = ['green1', 'green2', 'ir1', 'ir2', 'accX', 'accY', 'accZ']

# 批量转换
df[cols_to_convert] = df[cols_to_convert].astype(float)

# 检查转换后的类型
print(df.dtypes)

Date            str
Time            str
Duration        str
green1      float64
green2      float64
ir1         float64
ir2         float64
accX        float64
accY        float64
accZ        float64
dtype: object


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.ndimage import gaussian_filter1d
from matplotlib.figure import Figure 

# 1. 设定目标列与提取数据 
col1 = 'green1'
try:
    y1_raw = -df[col1].astype(float).values
except NameError:
    print("请先确保 df 已经被定义并加载了数据！")
    y1_raw = np.random.randn(5000) 

# 极简 Z-score 标准化
y1_mean = np.nanmean(y1_raw)
y1_std = np.nanstd(y1_raw)
y1 = (y1_raw - y1_mean) / (y1_std + 1e-8)  

fs = 100.0  

# ---------------------------------------------------------
# 2. 交互组件初始化 (新增两个 Sigma 控制器)
# ---------------------------------------------------------
start_slider = widgets.IntSlider(min=0, max=max(0, len(y1)-200), step=100, value=0, description='Start Idx', continuous_update=False)
window_slider = widgets.IntSlider(min=200, max=10000, step=200, value=1000, description='Window', continuous_update=False)

# sigma_base 用于提取宏观基线 (值很大)
sigma_base_slider = widgets.FloatSlider(value=50.0, min=10.0, max=200.0, step=5.0, description=r'$\sigma$ Baseline', continuous_update=False)
# sigma_pulse 用于去除微小毛刺 (值很小)
sigma_pulse_slider = widgets.FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5, description=r'$\sigma$ Pulse', continuous_update=False)

out = widgets.Output()

# ---------------------------------------------------------
# 3. 核心绘图函数：双趟高斯滤波提取法
# ---------------------------------------------------------
def update_plot(*args):
    start = start_slider.value
    window = window_slider.value
    sigma_base = sigma_base_slider.value
    sigma_pulse = sigma_pulse_slider.value
    
    end = min(start + window, len(y1))
    t = np.arange(start, end) / fs
    
    if start >= len(y1) or window <= 0: return
    current_y1 = y1[start:end]
    if len(current_y1) == 0: return
        
    # --- 核心算法：带通解耦 ---
    # 1. 提取极低频基线
    baseline = gaussian_filter1d(current_y1, sigma=sigma_base)
    # 2. 去趋势：原始信号减去基线，把信号拉平到 0 刻度
    detrended = current_y1 - baseline
    # 3. 提取纯净脉搏：对拉平后的信号进行轻微平滑，去除高频毛刺
    clean_pulse = gaussian_filter1d(detrended, sigma=sigma_pulse)
    
    # 顺便求个纯净的一阶导数备用
    pulse_derivative = gaussian_filter1d(detrended, sigma=sigma_pulse, order=1) * fs

    with out:
        clear_output(wait=True) 
        fig = Figure(figsize=(15, 10))
        
        # 图 1：原始信号 vs 提取出的缓慢基线
        ax1 = fig.add_subplot(311)
        ax1.plot(t, current_y1, '-', color='#2ca02c', linewidth=1.0, alpha=0.5, label=f'Raw {col1}')
        ax1.plot(t, baseline, '--', color='#1f77b4', linewidth=2.0, label=rf'Extracted Baseline ($\sigma$={sigma_base})')
        ax1.set_title(f'Step 1: Baseline Drift Extraction', fontweight='bold')
        ax1.legend(loc='upper right')
        ax1.grid(True, linestyle='--', alpha=0.4)
        
        # 图 2：去趋势后纯净的脉搏波 (核心结果)
        ax2 = fig.add_subplot(312, sharex=ax1)
        ax2.axhline(0, color='gray', linestyle='-', alpha=0.3)
        # 画出带有毛刺的去基线信号作为背景对比
        ax2.plot(t, detrended, '-', color='gray', linewidth=1.0, alpha=0.9, label='Detrended (Noisy)')
        # 画出最终极其平滑且无相位偏移的完美脉搏波
        ax2.plot(t, clean_pulse, '-', color='#d62728', linewidth=2.0, label=rf'Clean Pulse ($\sigma$={sigma_pulse})')
        ax2.set_title('Step 2: Clean Pulse Recovery (Zero Phase Shift)', fontweight='bold')
        ax2.set_ylabel('Amplitude', fontweight='bold')
        ax2.legend(loc='upper right')
        ax2.grid(True, linestyle='--', alpha=0.4)

        # 图 3：纯净脉搏波的一阶导数
        ax3 = fig.add_subplot(313, sharex=ax1)
        ax3.axhline(0, color='gray', linestyle='--', alpha=0.5)
        ax3.plot(t, pulse_derivative, '-', color='#9467bd', linewidth=1.5, label='Pulse Derivative (dx/dt)')
        ax3.set_xlabel('Time (s)')
        ax3.set_ylabel('Velocity', color='#9467bd', fontweight='bold')
        ax3.legend(loc='upper right')
        ax3.grid(True, linestyle='--', alpha=0.4)
        
        fig.tight_layout()
        display(fig)

start_slider.observe(update_plot, 'value')
window_slider.observe(update_plot, 'value')
sigma_base_slider.observe(update_plot, 'value')
sigma_pulse_slider.observe(update_plot, 'value')

ui = widgets.VBox([
    widgets.HBox([start_slider, window_slider]), 
    widgets.HBox([sigma_base_slider, sigma_pulse_slider]), 
    out
])

display(ui)
update_plot()

In [4]:
import numpy as np
from scipy.ndimage import gaussian_filter1d

def filter_ppg_gaussian(signal, sigma_base=50.0, sigma_pulse=2.0, fs=100.0, return_all=False):
    """
    双趟高斯滤波提取纯净 PPG 信号
    """
    # 1. 提取极低频基线
    baseline = gaussian_filter1d(signal, sigma=sigma_base)
    
    # 2. 去趋势
    detrended = signal - baseline
    
    # 3. 提取纯净脉搏
    clean_pulse = gaussian_filter1d(detrended, sigma=sigma_pulse)
    
    if return_all:
        pulse_derivative = gaussian_filter1d(detrended, sigma=sigma_pulse, order=1) * fs
        return clean_pulse, baseline, pulse_derivative
        
    return clean_pulse

In [5]:
col1 = 'green1'
try:
    y1_raw = -df[col1].astype(float).values
except NameError:
    print("请先确保 df 已经被定义并加载了数据！")

In [6]:
clean_pulse, baseline, pulse_derivative = filter_ppg_gaussian(y1_raw, sigma_pulse=4.5, return_all=True)

In [7]:
clean_pulse

array([ 9839.7063164 ,  9854.04443692,  9882.24274965, ...,
       -5481.75554887, -6286.11152107, -6695.70899647], shape=(581710,))

In [8]:
clean_pulse.mean()

np.float64(-9.724935865914986e-10)

In [9]:
y1_raw.mean()

np.float64(-8500497.296276495)

In [10]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

def interactive_ppg_viewer(raw_sig, clean_sig, fs=100.0):
    """
    分离显示上下距离的 PPG 信号对比查看器
    """
    max_idx = max(0, len(raw_sig) - 100)
    
    start_slider = widgets.IntSlider(
        min=0, max=max_idx, step=100, value=0, 
        description='起点 (Start)', 
        continuous_update=False, 
        layout=widgets.Layout(width='80%')
    )
    
    
    window_slider = widgets.IntSlider(
        min=100, max=10000, step=200, value=1000, 
        description='窗口 (Window)', 
        continuous_update=False, 
        layout=widgets.Layout(width='80%')
    )
    
    out = widgets.Output()
    
    def update_plot(change=None):
        start = start_slider.value
        window = window_slider.value
        end = min(start + window, len(raw_sig))
        
        t = np.arange(start, end) / fs
        
        with out:
            clear_output(wait=True)
            
            # 【核心修改】：创建上下两个子图，并设置 sharex=True 共享时间轴
            fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(14, 8), sharex=True)
            
            # 上图：原始信号
            ax1.plot(t, raw_sig[start:end], color='gray', alpha=0.8, label='Raw Signal', linewidth=1)
            ax1.set_title(f'PPG Comparison (Indices: {start} to {end})', fontweight='bold')
            ax1.set_ylabel('Raw Amplitude')
            ax1.legend(loc='upper right')
            ax1.grid(True, linestyle='--', alpha=0.4)
            
            # 下图：纯净信号
            ax2.plot(t, clean_sig[start:end], color='#d62728', label='Clean Pulse', linewidth=2)
            ax2.set_xlabel('Time (s)')
            ax2.set_ylabel('Clean Amplitude')
            ax2.legend(loc='upper right')
            ax2.grid(True, linestyle='--', alpha=0.4)
            
            # 缩减两个子图之间的空白
            fig.tight_layout()
            plt.subplots_adjust(hspace=0.1) 
            
            plt.show()
            plt.close(fig)
            
    start_slider.observe(update_plot, names='value')
    window_slider.observe(update_plot, names='value')
    
    ui = widgets.VBox([start_slider, window_slider, out])
    display(ui)
    update_plot()

# --- 运行调用 ---
interactive_ppg_viewer(y1_raw, clean_pulse, fs=100.0)

In [11]:
clean_pulse.min()

np.float64(-362936.2422976161)

In [12]:
clean_pulse.max()

np.float64(224675.68957972317)

In [13]:
clean_pulse.mean()

np.float64(-9.724935865914986e-10)